In [132]:
from gitsource import GithubRepositoryDataReader

In [133]:
from gitsource import chunk_documents

In [134]:
from embedder import Embedder

In [135]:
from minsearch import VectorSearch

In [136]:
from minsearch import Index

In [137]:
embedder = Embedder()

In [138]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)
len(v)

384

In [139]:
v[0]

np.float64(-0.02058203437252893)

In [140]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)


In [141]:
documents = [file.parse() for file in reader.read()]

In [142]:
len(documents)

72

In [143]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [144]:
for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        sqlite_doc = doc
        break

In [145]:
page_vector = embedder.encode(sqlite_doc["content"])

In [146]:
page_vector.dot(v)

np.float64(0.36107027225589694)

In [147]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [148]:
len(chunks)

295

In [149]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [150]:
X = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

In [151]:
X.shape

(295, 384)

In [152]:
scores = X.dot(v)

In [153]:
import numpy as np

In [154]:
best = scores.argmax()

In [155]:
chunks[best]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [156]:
vs = VectorSearch()

In [157]:
vs.fit(X, chunks)

In [158]:
query = "What metric do we use to evaluate a search engine?"

In [159]:
query

'What metric do we use to evaluate a search engine?'

In [160]:
query_vector = embedder.encode(query)

In [161]:
results = vs.search(query_vector, num_results=5)

In [162]:
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [163]:
for r in results:
    print(r["filename"])

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


In [164]:
query = "How do I store vectors in PostgreSQL?"

In [165]:
index = Index(
    text_fields=["content"]
)

In [166]:
index.fit(chunks)

In [167]:
query = "How do I store vectors in PostgreSQL?"

query_vector = embedder.encode(query)

vector_results = vs.search(
    query_vector,
    num_results=5
)

In [168]:
text_results = index.search(
    query=query,
    num_results=5
)

In [169]:
print("VECTOR SEARCH")
for r in vector_results:
    print(r["filename"])

VECTOR SEARCH
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [170]:
print("\nTEXT SEARCH")
for r in text_results:
    print(r["filename"])


TEXT SEARCH
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [171]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

In [172]:
query = "How do I give the model access to tools?"

In [173]:
query_vector = embedder.encode(query)

In [175]:
vector_results = vs.search(
    query_vector,
    num_results=5
)

In [176]:
text_results = index.search(
    query=query,
    num_results=5
)

In [177]:
results = rrf([vector_results, text_results])

In [178]:
for r in results:
    print(r["filename"])

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
